# 04 · Indicadores técnicos

**Objetivo:** calcular, para cada una de las 1.000 empresas y cada día de los 10 años de histórico, los 9 indicadores técnicos del proyecto. El cálculo vive en `funciones.py` (`calcular_indicadores`), porque el screener, el backtesting y la interfaz deben usar exactamente la misma fórmula. Este notebook lo aplica, lo enseña sobre un ejemplo y verifica que los resultados son coherentes.

**Entradas:**
- `data/cache/precios_historicos.parquet` (salida del notebook 02): OHLCV diario ajustado, 10 años, 1.000 empresas

**Salidas:**
- `data/cache/indicadores.parquet` — una fila por empresa y día con todos los indicadores (entrada de los notebooks 05 y 09)

In [1]:
import time

import numpy as np
import pandas as pd
import pandas_ta as ta
import plotly.graph_objects as go
from dotenv import load_dotenv
from plotly.subplots import make_subplots

from funciones import (
    CARPETA_CACHE,
    CMF_SESIONES,
    DONCHIAN_SESIONES,
    calcular_cmf,
    calcular_indicadores,
)

load_dotenv();

## 1. Precios de partida

In [2]:
precios = pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet")
precios = precios.sort_values(["ticker", "fecha"]).reset_index(drop=True)
print("Filas:", len(precios), "· Empresas:", precios["ticker"].nunique())
print("Fechas:", precios["fecha"].min().date(), "→", precios["fecha"].max().date())

Filas: 2281091 · Empresas: 1000
Fechas: 2016-10-10 → 2026-10-09


## 2. Los 9 indicadores y para qué sirve cada uno

| Indicador | Parámetros | Qué mide | Uso en el sistema |
|---|---|---|---|
| **RSI** | 14 sesiones | Fuerza de las subidas frente a las bajadas, de 0 a 100. Por debajo de 30 = sobreventa; por encima de 70 = sobrecompra | Señal de reversión en mercado lateral |
| **MACD** | 12 / 26 / 9 | Diferencia entre una media exponencial rápida y una lenta, su media (señal) y la diferencia entre ambas (histograma) | Contexto visual |
| **Bandas de Bollinger** | 20 sesiones, ±2σ | Media de 20 sesiones y dos bandas a 2 desviaciones típicas: el "rango normal" del precio | Contexto visual y rango de precio esperado |
| **Fibonacci dinámico** | Máx./mín. de 45 sesiones | Niveles de retroceso (23,6 / 38,2 / 50 / 61,8 / 78,6 %) entre el máximo y el mínimo recientes | Rango de precio esperado |
| **CMF** | 20 sesiones, umbral ±0,07 | Si el volumen acompaña a las subidas (+) o a las bajadas (−), de −1 a 1 | Confirmación del cruce de EMA |
| **ADX** | 14 sesiones, umbral 25 | Fuerza de la tendencia, sin importar si es alcista o bajista | Clasificar el régimen: tendencia o lateral |
| **Cruce de EMA** | EMA(4) y EMA(8) | Dos medias exponenciales, rápida y lenta; cuando la rápida cruza a la lenta, cambia la dirección a corto plazo | Señal en régimen de tendencia |
| **Canal de Donchian** | 20 sesiones | Máximo y mínimo de las 20 sesiones anteriores | Señal de ruptura en cualquier régimen |
| **ATR** | 14 sesiones | Rango medio diario del precio (volatilidad en dólares) | Base de todos los stops |

Las **umbrales** (30/70, 25, ±0,07) no se aplican aquí: este notebook solo calcula los indicadores. Las reglas de señal se aplican en el notebook 05.

## 3. Ejemplo con una sola empresa (Apple)

`calcular_indicadores` recibe los precios de **una** empresa ordenados por fecha y devuelve una columna por indicador:

In [3]:
precios_apple = precios[precios["ticker"] == "AAPL"].set_index("fecha")
indicadores_apple = calcular_indicadores(precios_apple)

print("Columnas:", list(indicadores_apple.columns))
indicadores_apple.tail(3).T.round(3)

Columnas: ['rsi', 'macd', 'macd_histograma', 'macd_senal', 'bollinger_inferior', 'bollinger_media', 'bollinger_superior', 'fibonacci_maximo', 'fibonacci_minimo', 'fibonacci_23_6', 'fibonacci_38_2', 'fibonacci_50_0', 'fibonacci_61_8', 'fibonacci_78_6', 'cmf', 'adx', 'ema_rapida', 'ema_lenta', 'donchian_minimo_previo', 'donchian_maximo_previo', 'atr']


fecha,2026-10-07,2026-10-08,2026-10-09
rsi,57.696,61.233,56.138
macd,3.282,3.561,3.437
macd_histograma,-0.821,-0.434,-0.446
macd_senal,4.103,3.995,3.883
bollinger_inferior,327.241,328.461,328.788
bollinger_media,334.478,335.171,335.389
bollinger_superior,341.715,341.880,341.990
fibonacci_maximo,345.340,345.340,345.340
fibonacci_minimo,300.570,300.570,300.570
fibonacci_23_6,334.774,334.774,334.774


Tres decisiones de cálculo que conviene conocer:

1. **Donchian "previo".** La regla de ruptura (sección 8.3 del CLAUDE.md) compara el cierre de hoy con el máximo de las **20 sesiones anteriores**. `pandas-ta` incluye el día de hoy en el canal, y como el cierre nunca puede superar el máximo del propio día, la ruptura no ocurriría nunca. Por eso el canal se desplaza un día (`shift(1)`).
2. **Bollinger con desviación típica poblacional (`ddof=0`).** Es la definición original de John Bollinger. `pandas-ta`, por defecto, usa la muestral (`ddof=1`), que da bandas algo más anchas.
3. **Fibonacci a mano.** No existe en `pandas-ta`. Cada nivel se mide desde el máximo de las últimas 45 sesiones hacia abajo: `nivel = máximo − porcentaje × (máximo − mínimo)`. Así, el nivel 61,8 % es un retroceso del 61,8 % de la subida reciente.

## 4. La CMF calculada a mano: por qué

Al calcular la CMF con `pandas-ta` sobre las 1.000 empresas aparecieron valores de **−22 y +30**, imposibles en un indicador que por definición va de −1 a 1. La causa son los días en que **máximo = mínimo**: la fórmula divide entre cero y `pandas-ta` lo "arregla" dividiendo entre un número diminuto.

Esos días aparecen en etapas de cotización casi nula: por ejemplo, Ferguson (`FERG`) en 2017, antes de trasladar su cotización principal a EEUU. Lo vemos con una sesión real:

In [4]:
precios_ferg = precios[precios["ticker"] == "FERG"].set_index("fecha")
dia = pd.Timestamp("2017-12-12")

cmf_pandas_ta = ta.cmf(precios_ferg["high"], precios_ferg["low"], precios_ferg["close"], precios_ferg["volume"], length=CMF_SESIONES)
cmf_propia = calcular_cmf(precios_ferg["high"], precios_ferg["low"], precios_ferg["close"], precios_ferg["volume"], CMF_SESIONES)

print(f"CMF de FERG el {dia.date()}: pandas-ta = {cmf_pandas_ta[dia]:.2f} · propia = {cmf_propia[dia]:.2f}")
precios_ferg.loc["2017-12-04":dia, ["open", "high", "low", "close", "volume"]]

CMF de FERG el 2017-12-12: pandas-ta = -22.08 · propia = 0.32


,open,high,low,close,volume
fecha,,,,,
2017-12-04,63.241135,63.241135,63.241135,63.241135,0.0
2017-12-05,63.241135,63.241135,63.241135,63.241135,0.0
2017-12-06,63.241135,63.241135,63.241135,63.241135,0.0
2017-12-07,61.944317,61.944317,61.944317,61.944317,1421.0
2017-12-08,61.866505,61.866505,61.866505,61.866505,379.0
2017-12-11,61.572567,61.572567,61.572567,61.572567,284.0
2017-12-12,60.863632,60.863632,60.863632,60.863632,2652.0


La convención habitual es tratar un día con máximo = mínimo como **neutro** (multiplicador 0): no hubo recorrido que indique presión compradora o vendedora. Si en las 20 sesiones no hubo ningún volumen, la CMF queda **vacía** (no hay flujo que medir) en lugar de inventar un valor.

En los días normales, la CMF propia coincide con la de `pandas-ta` (se comprueba al final). Las etapas sin liquidez también afectan a otros indicadores; se tratarán en el notebook 05 exigiendo un volumen mínimo para generar señales.

## 5. Cálculo para las 1.000 empresas

Se aplica la misma función a cada empresa por separado: los indicadores de una empresa nunca deben mezclar precios de otra.

In [5]:
inicio_calculo = time.time()
tablas = []
for ticker, precios_ticker in precios.groupby("ticker"):
    indicadores_ticker = calcular_indicadores(precios_ticker.set_index("fecha"))
    indicadores_ticker.insert(0, "ticker", ticker)
    tablas.append(indicadores_ticker.reset_index())

indicadores = pd.concat(tablas, ignore_index=True)
print(f"Calculado en {time.time() - inicio_calculo:.1f} s · filas: {len(indicadores)} · columnas: {indicadores.shape[1]}")

Calculado en 5.1 s · filas: 2281091 · columnas: 23


## 6. Gráfico de comprobación (Apple, último año)

Un vistazo visual para comprobar que cada indicador hace lo que debe: las bandas envuelven el precio, el RSI sube en las subidas, el ADX crece cuando hay tendencia clara… Los niveles de Fibonacci se dibujan con los valores del último día.

In [6]:
ultimo_anio = indicadores_apple.index >= indicadores_apple.index.max() - pd.DateOffset(years=1)
p, i = precios_apple[ultimo_anio], indicadores_apple[ultimo_anio]

grafico = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                        row_heights=[0.44, 0.14, 0.14, 0.14, 0.14],
                        subplot_titles=["AAPL: precio, Bollinger, EMA, Donchian y Fibonacci", "RSI", "MACD", "ADX", "CMF"])

grafico.add_trace(go.Candlestick(x=p.index, open=p["open"], high=p["high"], low=p["low"], close=p["close"], name="Precio"), row=1, col=1)
for columna, estilo in [("bollinger_superior", "dot"), ("bollinger_media", "dash"), ("bollinger_inferior", "dot")]:
    grafico.add_trace(go.Scatter(x=i.index, y=i[columna], name=columna, line={"dash": estilo, "width": 1}), row=1, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["ema_rapida"], name="EMA 4", line={"width": 1}), row=1, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["ema_lenta"], name="EMA 8", line={"width": 1}), row=1, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["donchian_maximo_previo"], name="Donchian máx. previo", line={"shape": "hv", "width": 1}), row=1, col=1)
for nivel in ["fibonacci_23_6", "fibonacci_38_2", "fibonacci_50_0", "fibonacci_61_8", "fibonacci_78_6"]:
    grafico.add_hline(y=i[nivel].iloc[-1], line_dash="dot", line_width=1, annotation_text=nivel, row=1, col=1)

grafico.add_trace(go.Scatter(x=i.index, y=i["rsi"], name="RSI"), row=2, col=1)
for umbral in [30, 70]:
    grafico.add_hline(y=umbral, line_dash="dash", line_width=1, row=2, col=1)
grafico.add_trace(go.Bar(x=i.index, y=i["macd_histograma"], name="Histograma MACD"), row=3, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["macd"], name="MACD", line={"width": 1}), row=3, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["macd_senal"], name="Señal MACD", line={"width": 1}), row=3, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["adx"], name="ADX"), row=4, col=1)
grafico.add_hline(y=25, line_dash="dash", line_width=1, row=4, col=1)
grafico.add_trace(go.Scatter(x=i.index, y=i["cmf"], name="CMF"), row=5, col=1)
for umbral in [-0.07, 0.07]:
    grafico.add_hline(y=umbral, line_dash="dash", line_width=1, row=5, col=1)

grafico.update_layout(height=1100, xaxis_rangeslider_visible=False, showlegend=False)
grafico.show(renderer="notebook_connected")

## 7. Resumen de los indicadores en todo el universo

Distribución de cada indicador sobre todas las empresas y días. Sirve para detectar valores imposibles (un RSI de 140, una CMF de 5…). Los indicadores en dólares (MACD, ATR, Bollinger…) dependen del precio de cada acción y no son comparables entre empresas.

In [7]:
indicadores[["rsi", "adx", "cmf", "macd_histograma", "atr"]].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(3)

,count,mean,std,min,1%,25%,50%,75%,99%,max
rsi,2279915.0,52.038,12.715,0.000,22.130,43.428,52.175,60.922,79.801,100.000
adx,2268020.0,24.305,10.175,0.098,9.625,16.865,22.180,29.599,55.644,100.000
cmf,2261744.0,0.016,0.158,-1.000,-0.352,-0.089,0.015,0.121,0.381,1.000
macd_histograma,2248091.0,-0.002,1.652,-176.043,-3.973,-0.226,0.004,0.238,3.802,99.554
atr,2268091.0,3.266,6.806,0.000,0.091,0.801,1.651,3.549,23.618,308.876


## 8. Guardar la salida

Se guardan solo las claves (`fecha`, `ticker`) y los indicadores; los precios ya están en `precios_historicos.parquet`. El notebook 05 une ambas tablas.

In [8]:
ruta_indicadores = CARPETA_CACHE / "indicadores.parquet"
indicadores.to_parquet(ruta_indicadores, index=False)
print(f"Guardado: {ruta_indicadores.name} - {len(indicadores)} filas - {ruta_indicadores.stat().st_size / 1e6:.0f} MB")

Guardado: indicadores.parquet - 2281091 filas - 282 MB


## Comprobaciones

Además de los rangos de valores, se verifican **contra un cálculo independiente** los tres indicadores con alguna parte propia: el canal de Donchian desplazado, la CMF propia y el ATR (base de todos los stops).

In [9]:
columnas_indicadores = [c for c in indicadores.columns if c not in ("fecha", "ticker")]

# Una fila de indicadores por cada fila de precios, mismas claves
assert len(indicadores) == len(precios)
assert indicadores[["fecha", "ticker"]].equals(precios[["fecha", "ticker"]])
assert not np.isinf(indicadores[columnas_indicadores]).any().any(), "Hay valores infinitos"

# Rangos teóricos (con una tolerancia mínima: el redondeo da a veces 100,00000000000001)
TOLERANCIA = 1e-9
assert indicadores["rsi"].dropna().between(-TOLERANCIA, 100 + TOLERANCIA).all()
assert indicadores["adx"].dropna().between(-TOLERANCIA, 100 + TOLERANCIA).all()
assert indicadores["cmf"].dropna().between(-1 - TOLERANCIA, 1 + TOLERANCIA).all(), "CMF fuera de [-1, 1]"
assert (indicadores["atr"].dropna() >= 0).all()

# Orden lógico de bandas y niveles
completas = indicadores.dropna()
assert (completas["bollinger_inferior"] <= completas["bollinger_media"]).all()
assert (completas["bollinger_media"] <= completas["bollinger_superior"]).all()
niveles_fibonacci = ["fibonacci_maximo", "fibonacci_23_6", "fibonacci_38_2", "fibonacci_50_0",
                     "fibonacci_61_8", "fibonacci_78_6", "fibonacci_minimo"]
for superior, inferior in zip(niveles_fibonacci, niveles_fibonacci[1:]):
    assert (completas[superior] >= completas[inferior]).all(), f"{superior} < {inferior}"

# Tras el periodo de arranque (60 sesiones) solo puede faltar la CMF, y solo si no hubo volumen en 20 sesiones
sesion = indicadores.groupby("ticker").cumcount()
tras_arranque = indicadores[sesion >= 60]
huecos = tras_arranque[columnas_indicadores].isna()
assert not huecos.drop(columns="cmf").any().any(), "Hay indicadores vacíos tras el arranque"
volumen_20 = precios.groupby("ticker")["volume"].transform(lambda v: v.rolling(CMF_SESIONES).sum())
assert (volumen_20[tras_arranque.index[huecos["cmf"]]] == 0).all()

# Donchian previo = máximo de las 20 sesiones anteriores, calculado de forma independiente
maximo_previo = precios.groupby("ticker")["high"].transform(lambda h: h.rolling(DONCHIAN_SESIONES).max().shift(1))
assert np.allclose(indicadores["donchian_maximo_previo"], maximo_previo, equal_nan=True)

# CMF propia = CMF de pandas-ta en los días sin máximo = mínimo (Apple)
dias_normales = (precios_apple["high"] > precios_apple["low"]).rolling(CMF_SESIONES).min() == 1
cmf_ta_apple = ta.cmf(precios_apple["high"], precios_apple["low"], precios_apple["close"], precios_apple["volume"], length=CMF_SESIONES)
assert np.allclose(indicadores_apple["cmf"][dias_normales], cmf_ta_apple[dias_normales])

# ATR a mano (rango verdadero + media de Wilder) = ATR de pandas-ta (Apple, tras 300 sesiones de arranque)
cierre_anterior = precios_apple["close"].shift(1)
rango_verdadero = pd.concat([precios_apple["high"] - precios_apple["low"],
                             (precios_apple["high"] - cierre_anterior).abs(),
                             (precios_apple["low"] - cierre_anterior).abs()], axis=1).max(axis=1)
atr_a_mano = rango_verdadero.ewm(alpha=1 / 14, adjust=False).mean()
assert np.allclose(atr_a_mano.iloc[300:], indicadores_apple["atr"].iloc[300:], rtol=1e-6)

print("Todas las comprobaciones OK")

Todas las comprobaciones OK
